In [ ]:
# Pytorch library for matrix (tensor) operations, matplotlib for visualizing the data, rand for (drumroll ...) randomness.
import matplotlib.pyplot as plot
import random
import torch
import torch.nn.functional as torchf

%matplotlib inline

In [ ]:
# There's going to be a significant number of floating point calculations, so using a GPU will be much faster! 
device = torch.device("cpu")
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Running calculations on metal")
elif torch.backends.cuda.is_availabile():
    device = torch.device("cuda")
    print("Running calculations on cuda")
else:
    print("Using the cpu for calcluations will be slow")

In [ ]:
# The dataset is a list of the top baby names in lowercase.
words = open("names_short.txt", "r").read().splitlines()
for (index, word) in enumerate(words):
    words[index] = word.lower()
len(words)

In [ ]:
words[:10]

In [ ]:
# Lookup tables used to convert our input to a numerical representation and back.
#
# We use a sorted set of all characters as the vocabulary. Then create a mapping from elements in the vocabulary to their index
# and another mapping in reverse.
# NOTE: The special character '.' is added to specify the start and end of a word.

input_vocab = sorted(list(set(".".join(words))))
stoi = { s: i for i, s in enumerate(input_vocab) }
itos = { i: s for s, i in stoi.items() }

In [ ]:
# We'll use a simple bigram counting probability method as the base case for model performance. 

# First we'll need a count of the bigrams (i.e. pair of letters, input letter and next or output letter).
bigram_counts = torch.zeros((len(input_vocab), len(input_vocab)), dtype=torch.int32, device=device)

# Create combinations of input letter + next letter and count the occurances.
for word in words:
    chars = ["."] + list(word) + ["."]
    # Loop over two lists, the characters and the characters offset by one (i.e. input -> target).
    for input_letter, output_letter in zip(chars, chars[1:]):
        bigram_counts[stoi[input_letter], stoi[output_letter]] += 1

In [ ]:
# Visualization of the the bigram counts matrix with a heatmap. 
bigram_counts_cpu = bigram_counts.cpu()
plot.figure(figsize=(16,16))
plot.imshow(bigram_counts_cpu, cmap='GnBu')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plot.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plot.text(j, i, bigram_counts_cpu[i, j].item(), ha="center", va="top", color='gray')
plot.axis('off');

In [ ]:
# Let's normalize the counts into probabilities and use that as a lookup table to predict the next character.
probabilities = bigram_counts / bigram_counts.sum(1, keepdim=True)

for _ in range(10):
    output = ""
    position = 0

    while True:
        output_probabilities = probabilities[position]
        position = torch.multinomial(output_probabilities, num_samples=1, replacement=True).item()

        if position == 0:
            break;

        output += itos[position]

    print(output);


In [ ]:
# That's not much better than nonsense, so let's try a simple neural network.

# For a neural network, we need to divide our data into two sets for training, a list of inputs and a list of outputs.
input_indices, output_indices = [], []

# Create combinations of input letter + next letter and count the occurances.
for word in words:
    chars = ["."] + list(word) + ["."]
    # Loop over two lists, the characters and the characters offset by one (i.e. input -> target).
    for input_letter, output_letter in zip(chars, chars[1:]):
        input_indices.append(stoi[input_letter])
        output_indices.append(stoi[output_letter])

# Convert the lists to tensors for processing
inputs = torch.tensor(input_indices, device=device)
outputs = torch.tensor(output_indices, device=device)

In [ ]:
# The inputs will be fed into the neural network with "one-hot" encodings, which just means each input letter is expanded
# to a tensor the length of the vocabulary that is all zeros with the letter index set to one, i.e. the tensor is hot or
# activated on the input letter. As an example, the following plot shows "liam" and "olivia" encoded as one-hots.
encoded = torchf.one_hot(inputs[:13], num_classes=27).float().to(device)
plot.imshow(encoded.cpu())

In [ ]:
# This simple neural network will have 27 "neurons" one for each letter in the vocabulary. The main representation of the
# network will be the weights, a matrix of values that represent the transformation of inputs to outputs. With 27 neurons,
# the weights will be a 27 x 27 matrix.
weights = torch.randn((27, 27), requires_grad=True, device=device)

print(f"Total number of model parameters {weights.nelement()} a.k.a. {weights.nelement()/1000000000:.6f} B")

In [ ]:
# Training runs.
#
# During each training round a forward pass and backward pass are run, the results of which are used to adjust the values
# of the weights. A forward pass is encoding the inputs as a one-hot tensor, matrix multiplying the encoded inputs with the
# weights, then normalizing the results to get a matrix of probabilities that has the shape of the inputs size by the
# vocabulary size. Then the loss, or an approximation of how far off the model is from the training data is calculated and
# used to adjust the weights. This example uses a log mean function with a weight decay.
for run in range(10000):
    # Forward pass.
    encodings = torchf.one_hot(inputs, num_classes=27).float()
    log_counts = (encodings @ weights).exp()
    probabilities = log_counts / log_counts.sum(1, keepdims=True)
    loss = -probabilities[torch.arange(inputs.nelement(), device=device), outputs].log().mean() + 0.01*(weights**2).mean()

    # Backward pass.
    weights.grad = None
    loss.backward()

    # Update the weights with the propagated loss.
    weights.data += -10 * weights.grad

In [ ]:
# After training the model, it can be sampled in a way similar to a forward pass to generate new words.
#
# In order to sample from the trained weights, a starting postion is encoded as a one-hot tensor and used
# to calculate a matrix of probabilities, similar to the forward pass above. We can then use the matrix of
for _ in range(10):
    output = ""
    position = 0

    while True:
        encodings = torchf.one_hot(torch.tensor([position], device=device), num_classes=27).float()
        log_counts = (encodings @ weights).exp()
        probabilities = log_counts / log_counts.sum(1, keepdims=True)

        position = torch.multinomial(probabilities, num_samples=1, replacement=True).item()
        if position == 0:
            break

        output += itos[position]

    print(output)

In [ ]:
# All those calculations and it wasn't any better than the counting method :(
# Let's make it more complicated with a Multi Layer Perceptron (MLP)!!

In [ ]:
# Model hyperparameters
block_size = 3
vocab_size = len(stoi)
embedding_size = 10
hidden_size = 1000
learning_rate = 0.1

In [ ]:
# The inputs now have a context window of length block_size to a single next expected letter.

# Building the dataset ...
input_indices, output_indices = [], []

for word in words:
    context = [0] * block_size

    for letter in word + ".":
        index = stoi[letter]
        input_indices.append(context)
        output_indices.append(index)
        context = context[1:] + [index]

inputs = torch.tensor(input_indices, device=device)
outputs = torch.tensor(output_indices, device=device)

In [ ]:
# Model parameters
C = torch.randn((vocab_size, embedding_size), device=device)
weights_1 = torch.randn((embedding_size * block_size, hidden_size), device=device) * (5/3) / ((embedding_size * block_size)**0.5)
biases_1 = torch.randn(hidden_size, device=device) * 0.01
weights_2 = torch.randn(hidden_size, vocab_size, device=device) * 0.01
biases_2 = torch.zeros(vocab_size, device=device)

# All the model parameters
parameters = [C, weights_1, biases_1, weights_2, biases_2]

# Turn on backpropagation gradients for all parameters.
for parameter in parameters:
    parameter.requires_grad = True

total = sum(parameter.nelement() for parameter in parameters)
print(f"Total number of model parameters {total} a.k.a. {total/1000000000:.6f} B")

In [ ]:
C.shape, weights_1.shape, biases_1.shape, weights_2.shape, biases_2.shape

In [ ]:
# Keep track of losses over all training rounds.
losses = []

In [ ]:
# Run training
iterations = 10000
for iteration in range(iterations):
    # Forward pass
    embeddings = C[inputs]
    h_pre_activation = embeddings.view((embeddings.shape[0], -1)) @ weights_1 + biases_1
    h = torch.tanh(h_pre_activation)

    log_counts = h @ weights_2 + biases_2
    loss = torchf.cross_entropy(log_counts, outputs)

    # Backward pass
    for parameter in parameters:
        parameter.grad = None

    loss.backward()

    # Update the weights
    # factor = learning_rate if iteration < iterations / 2 else learning_rate / 10
    factor = learning_rate

    for parameter in parameters:
        parameter.data += -factor * parameter.grad

    losses.append(loss.item())

In [ ]:
sum, count = 0, 0
for loss in losses[len(losses)-100:]:
    sum += loss
    count += 1

sum / count

In [ ]:
plot.plot(losses)

In [ ]:
# visualize dimensions 0 and 1 of the embedding matrix C for all characters
C_cpu = C.cpu()
plot.figure(figsize=(8,8))
plot.scatter(C_cpu[:,0].data, C_cpu[:,1].data, s=200)
for i in range(C_cpu.shape[0]):
    plot.text(C_cpu[i,0].item(), C_cpu[i,1].item(), itos[i], ha="center", va="center", color='white')
plot.grid('minor')

In [ ]:
# Sample from the model
for _ in range(20): 
    output = ""
    context = [0] * block_size
   
    while True:
        embeds = C[torch.tensor([context], device=device)]
        h = torch.tanh(embeds.view(embeds.shape[0], -1) @ weights_1 + biases_1)
        logits = h @ weights_2 + biases_2
        probabilities = torchf.softmax(logits, dim=1)

        position = torch.multinomial(probabilities, num_samples=1).item()
        if position == 0:
            break

        context = context[1:] + [position]
        output += itos[position]

    print(output)

In [ ]:
# That's starting to look more like names, mathmatical!